In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sql_f

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ActiveLearning")
    .getOrCreate()
)

# Obtenemos el SparkContext asociado a la SparkSession
sc = spark.sparkContext

## PREPROCESAMIENTO NECESARIO

In [2]:
# Lectura del dataframe
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv("data/susy-10.csv")
)

In [ ]:
# Cast columns
df = (
    df
    .withColumn("id_sample", sql_f.col("id_sample").cast("int"))
    .withColumn("label", sql_f.col("label").cast("int"))
)

feature_cols = [col_name for col_name in df.columns if col_name.startswith("feature_")]

for col_name in feature_cols:
    df = df.withColumn(col_name, sql_f.col(col_name).cast("float"))



root
 |-- id_sample: integer (nullable = true)
 |-- label: integer (nullable = true)
 |-- feature_1: float (nullable = true)
 |-- feature_2: float (nullable = true)
 |-- feature_3: float (nullable = true)
 |-- feature_4: float (nullable = true)
 |-- feature_5: float (nullable = true)
 |-- feature_6: float (nullable = true)
 |-- feature_7: float (nullable = true)
 |-- feature_8: float (nullable = true)
 |-- feature_9: float (nullable = true)
 |-- feature_10: float (nullable = true)
 |-- feature_11: float (nullable = true)
 |-- feature_12: float (nullable = true)
 |-- feature_13: float (nullable = true)
 |-- feature_14: float (nullable = true)
 |-- feature_15: float (nullable = true)
 |-- feature_16: float (nullable = true)
 |-- feature_17: float (nullable = true)
 |-- feature_18: float (nullable = true)



In [4]:
# Convertir las features a un vector con VectorAssembler para poder aplicar el modelo
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# Se aplica sobre todo el conjunto de datos
df = assembler.transform(df)

# Se eliminan las columnas feature_cols
df = df.drop(*feature_cols)
df.show(1, truncate=False)

+---------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                 |
+---------+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
print(
    "Original DataFrame partitions:",
    df.rdd.getNumPartitions()
)

Original DataFrame partitions: 9


In [6]:
# Dividir en train/test
from config import TRAIN_TEST_SPLIT
from config import RANDOM_SEED
# División train/test
train_df, test_df = df.randomSplit(
    TRAIN_TEST_SPLIT,
    seed=RANDOM_SEED
)

In [7]:
# Seleccionar aleatoriamente conjunto etiquetado (L) y conjunto no etiquetado (U)
from config import INITIAL_LABELED_FRACTION

train_df = train_df.withColumn(
    "state",
    sql_f.when(
        sql_f.rand(RANDOM_SEED) < INITIAL_LABELED_FRACTION,
        "L"
    ).otherwise("U")
)

In [8]:
# Se almacena en cache tanto train_df como test_df
train_df.cache()
test_df.cache()

DataFrame[id_sample: int, label: int, features: vector]

In [9]:
# Se materializa la cache de train_df y se cuenta tanto labeled como unlabeled
stats = train_df.select(
    sql_f.count(sql_f.when(sql_f.col("state") == "L", True)).alias("labeled"),
    sql_f.count(sql_f.when(sql_f.col("state") == "U", True)).alias("unlabeled"),
).first()

# Extraer los valores
labeled_size = stats["labeled"]
unlabeled_size = stats["unlabeled"]
train_size = labeled_size + unlabeled_size
print(train_size)
print(labeled_size)
print(unlabeled_size)

70262
3490
66772


In [10]:
# Se materializa la cache de test_df
test_size = test_df.count()
print(test_size)

29738


In [11]:
print(
    "Original DataFrame partitions:",
    df.rdd.getNumPartitions()
)
print(
    "Train DataFrame partitions:",
    train_df.rdd.getNumPartitions()
)
print(
    "Test DataFrame partitions:",
    test_df.rdd.getNumPartitions()
)

Original DataFrame partitions: 9
Train DataFrame partitions: 9
Test DataFrame partitions: 9


## ENTRENAMIENTO INICIAL DEL CLASIFICADOR

In [12]:
# Entrenar clasificador LogisticRegresion de MLLlib para calcular probabilidades
from pyspark.ml.classification import LogisticRegression

# Crear clasificador
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability"
)

In [13]:
# Conjunto de datos etiquetado
labeled_df = train_df.filter(
    sql_f.col("state") == "L"
)

# Entrenar el modelo
lr_model = lr.fit(labeled_df)

In [14]:
print(
    "Labeled DataFrame partitions:",
    labeled_df.rdd.getNumPartitions()
)

Labeled DataFrame partitions: 9


In [15]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Realizar predicciones
test_predictions = lr_model.transform(test_df)

# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(test_predictions)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7891


## INICIO DEL ALGORITMO ITERATIVO

**CÁLCULO DE LAS PREDICCIONES SOBRE EL CONJUNTO NO ETIQUETADO (U)**

Para clasificación binaria con Logistic Regression, una medida muy sencilla y habitual es la incertidumbre basada en la distancia a 0.5:
$$uncertainty=1−∣P(y=1)−0.5∣×2$$

Con esta formula la incertidumbre maxima se da en P(Y=1)=0.5 de modo que la incertidumbre es 1.

In [16]:
unlabeled_df = train_df.filter(sql_f.col("state") == "U")
unlabeled_pred_df = lr_model.transform(unlabeled_df)
unlabeled_pred_df.show(5, truncate=False)

+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------------------------------------+----------------------------------------+----------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                    |state|rawPrediction                           |probability                             |prediction|
+---------+-----+---------------------------------------------------

In [17]:
# Calculo de incertidumbre
from pyspark.ml.functions import vector_to_array

unlabeled_pred_df = unlabeled_pred_df.withColumn(
    "uncertainty",
    1 - 2 * sql_f.abs(
        vector_to_array("probability")[1] - 0.5
    )
)
unlabeled_pred_df.show(5,truncate=False)

+---------+-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------------------------------------+----------------------------------------+----------+-------------------+
|id_sample|label|features                                                                                                                                                                                                                                                                                                                                                    |state|rawPrediction                           |probability                             |prediction|uncertainty        |
+---------+-----+-----------

**Filtrado inicial de candidatos con mayor incertidumbre**

In [18]:
from config import QUERY_BATCH_FRACTION
from config import UNCERTAINTY_CANDIDATES_MULTIPLIER


# Aqui hay que hacer unos prints informativos en los que
# se muestra el porcentaje de cada cosa y se entienda

# Tamaño del conjunto no etiquetado
# Esta variable es mejor obtenerla al inicio en el count
# una sola vez
unlabeled_size = unlabeled_pred_df.count()

# No podemos pedir mas muestras al oráculo de las que quedan en U
# Hacer la operacion train_size * Query_BATCH_FRACTION al inicio
# almacenar en una variable llamada B y mostrar en configuracion
query_batch = min(
    int(train_size * QUERY_BATCH_FRACTION),
    unlabeled_size
)
p = min(1.0, (UNCERTAINTY_CANDIDATES_MULTIPLIER * query_batch) / unlabeled_size)
print(f"Training samples: {train_size}")
print(f"Unlabeled samples: {unlabeled_size}")
print(f"Query batch: {query_batch}")
print(f"Proporción de candidatos a seleccionar en fase de incertidumbre (p): {p*100}%")

Training samples: 70262
Unlabeled samples: 66772
Query batch: 702
Proporción de candidatos a seleccionar en fase de incertidumbre (p): 10.513388845623913%


In [19]:
quantile_target = 1.0 - p
print(quantile_target)

0.8948661115437608


In [ ]:
# 3. Cálculo del umbral distribuido mediante approxQuantile (Greenwald-Khanna)
EPSILON = 0.001
threshold_val = unlabeled_pred_df.stat.approxQuantile(
    "uncertainty", [1.0 - p], EPSILON
)[0]

# 4. Filtrado distribuido 
uncertainty_candidates_df = unlabeled_pred_df.select("id_sample","features","uncertainty").filter(sql_f.col("uncertainty") >= threshold_val)

In [21]:
print(
    "Uncertainty candidates DataFrame partitions:",
    uncertainty_candidates_df.rdd.getNumPartitions()
)

Uncertainty candidates DataFrame partitions: 9


In [22]:
uncertainty_candidates_df.count()

7087

In [23]:
uncertainty_candidates_df.show(5, truncate=False)

+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|id_sample|features                                                                                                                                                                                                                                                                                                                                                  |uncertainty       |
+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**DIVERSITY CON K-MEANS**

In [24]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans(
    k=query_batch,
    seed=RANDOM_SEED,
    featuresCol="features",
    predictionCol="cluster_id",
    maxIter=20,
)

kmeans_model = kmeans.fit(uncertainty_candidates_df)

Esto no recoge los candidatos en el driver. KMeans.fit() utiliza la implementación distribuida de Spark.

Después:

In [25]:
clustered_candidates_df = kmeans_model.transform(uncertainty_candidates_df)

Cada fila queda asociada al cluster cuyo centroide le corresponde:

In [26]:
clustered_candidates_df.show(5, truncate=False)

+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+----------+
|id_sample|features                                                                                                                                                                                                                                                                                                                                                  |uncertainty       |cluster_id|
+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Calculo de la distancia al centroide correspondiente para seleccionar en cada cluster el mas cercano al centroide**

In [27]:
centers_broadcast = spark.sparkContext.broadcast(kmeans_model.clusterCenters())

Esta variable contiene para cada cluster, el centroide con las 18 caracteristicas.

In [28]:
centers = centers_broadcast.value

print("Number of cluster centers:", len(centers))
print("Number of features:", len(centers[0]))

print("\nFirst 5 cluster centers:")
for cluster_id, center in enumerate(centers[:5]):
    print(f"Cluster {cluster_id}: {center}")

Number of cluster centers: 702
Number of features: 18

First 5 cluster centers:
Cluster 0: [ 0.92234697 -0.00687305 -0.91170109  1.08647486 -0.27415058  0.91517304
  0.78998222 -0.7349836   0.84336312  0.24495323  0.85163637  0.88984231
  0.95203721  1.04331367  0.84689156  0.84845498  0.9928081   0.1585825 ]
Cluster 1: [ 0.83814603  1.2604676  -1.53072702  0.7847471   0.60310159  0.16625502
  0.80808599  0.1847179   0.17458349 -0.11996957  0.75822253  0.88357937
  1.06383154  0.18344626  0.78056399  0.50361221  1.13122743  0.37916854]
Cluster 2: [ 0.85409329  0.57897097  0.2094591   1.00182254  0.38617162 -0.45730515
  0.81942647 -1.54984945  0.93744396 -0.32404003  0.78489131  1.09136842
  1.24345019  2.10524144  0.72005602  1.58283647  0.43990914  0.2939451 ]
Cluster 3: [ 0.74034813  1.15678826 -0.27201258  0.77876296  1.63917287 -0.51337467
  1.38725325  1.2615787   2.05788023 -0.95363488  0.68128638  1.35209829
  1.77584223  2.66564103  0.77601503  2.02061177  1.22510471  0.370329

Ahora hay que obtener en cada cluster la muestra más cercana al centroide. Se opta por una solución basada en RDD porque:
- En lugar de enviar $N$ muestras (millones de filas) por la red, Spark filtra primero localmente en cada partición ($P$). Por la red solo viaja 1 muestra candidata por cluster ($K$) por cada partición. El volumen de datos transferidos pasa de gigabytes a apenas unos kilobytes.
- Vectorización y acceso $O(1)$:Al usar centers_broadcast.value[cluster_id], cada executor accede en memoria local al array de centroides de forma instantánea sin peticiones remotas.

**Paso 1: Mapeo a Estructura Par (Clave, Valor)**
Spark requiere que los RDDs tengan la estructura de tupla de dos elementos (K, V) para aplicar transformaciones basadas en claves como reduceByKey.
- Clave (K): row.cluster_id. Agrupa todas las muestras que pertenecen al mismo cluster.
- Valor (V): Una tupla interna (id_sample, distance).
- Cálculo de Distancia:
centers_broadcast.value[row.cluster_id] accede en $O(1)$ a los centroides descargados localmente en cada executor. Usamos Vectors.squared_distance para evitar la raíz cuadrada (la distancia al cuadrado preserva el orden del mínimo y es numéricamente más rápida).

In [31]:
from pyspark.ml.linalg import Vectors
# 1. Transformar DataFrame a RDD estructurado (clave, valor)
rdd_mapped = clustered_candidates_df.rdd.map(lambda row: (
    row.cluster_id,  # CLAVE: Identificador del cluster
    (
        row.id_sample,  # VALOR: (id de la muestra,
        float(Vectors.squared_distance(row.features, centers_broadcast.value[row.cluster_id]))  # distancia al centroide)
    )
))

La estructura del elemento resultante es:
( cluster_id , ( id_sample , distance ) )

In [35]:
sample_mapped = rdd_mapped.take(5)
sample_mapped
for item in sample_mapped:
    cluster_id = item[0]
    id_sample, distance = item[1]
    print(f"Cluster: {cluster_id} | Sample ID: {id_sample} | Distancia: {distance:.4f}")

Cluster: 170 | Sample ID: 44 | Distancia: 1.3962
Cluster: 161 | Sample ID: 59 | Distancia: 0.7756
Cluster: 619 | Sample ID: 68 | Distancia: 0.8415
Cluster: 494 | Sample ID: 73 | Distancia: 1.7404
Cluster: 221 | Sample ID: 84 | Distancia: 0.9109


**Paso 2: Agregación con reduceByKey**

reduceByKey recibe una función reductora (V, V) -> V que toma dos valores asociados a la misma clave y devuelve el "ganador".
- candidate1 es una tupla (id_sample_A, dist_A).
- candidate2 es una tupla (id_sample_B, dist_B).
- candidate1[1] accede a la distancia. 
Si la distancia de $A$ es menor que la de $B$, $A$ se conserva; de lo contrario, se conserva $B$.

¿Cómo funciona internamente? (Map-Side Combine)En lugar de enviar todas las filas por la red (shuffle), Spark ejecuta este lambda localmente en cada partición. Si una partición tiene $100.000$ filas repartidas en $10$ clusters, el executor reduce localmente esas $100.000$ filas a solo $10$ tuplas (una por cluster) antes de enviarlas por la red para la agregación final.

In [36]:
closest_per_cluster_rdd = rdd_mapped.reduceByKey(
    lambda candidate1, candidate2: candidate1 if candidate1[1] < candidate2[1] else candidate2
)

El resultado vuelve a ser el formato:

( cluster_id , ( id_sample , distance ) )

In [ ]:
results = closest_per_cluster_rdd.take(5)

for cluster_id, (id_sample, dist) in sorted(results, key=lambda x: x[0]):
    print(f"Cluster {cluster_id:2d} -> id_sample más cercano: {id_sample:8d} (Distancia: {dist:.6f})")

Total de clusters procesados: 5

Cluster  0 -> id_sample más cercano:    17171 (Distancia: 0.340858)
Cluster 36 -> id_sample más cercano:    26351 (Distancia: 0.567124)
Cluster 99 -> id_sample más cercano:    14329 (Distancia: 0.485185)
Cluster 198 -> id_sample más cercano:    69865 (Distancia: 0.545412)
Cluster 333 -> id_sample más cercano:    86981 (Distancia: 0.521872)


In [39]:
print(
    "Number of partitions:",
    closest_per_cluster_rdd.getNumPartitions()
)

Number of partitions: 9


cada elemento del RDD tiene esta estructura:

(cluster_id, (id_sample, distance))

In [40]:
closest_per_cluster_rdd.count()

702

## FASE FINAL: ACTUALIZACIÓN DE ETIQUETAS EN DATAFRAME

Este bloque final toma el RDD filtrado con los candidatos más cercanos de cada cluster, lo transforma en un DataFrame y actualiza las filas correspondientes en train_df.

**Primer paso: Extracción de IDs y creación de target_ids_df**

Estructura de origen: Cada elemento x en closest_per_cluster_rdd tiene la forma:
$$\text{x} = (\text{cluster\_id},\ (\text{id\_sample},\ \text{distance}))$$

El indexado de esta estructura es:
- x[0] $\rightarrow$ cluster_idx
- x[1] $\rightarrow$ (id_sample, distance)
- x[1][0] $\rightarrow$ id_sample (extrae solo el identificador de la muestra).

La coma (x[1][0],): En Python, (valor,) define una tupla de un solo elemento. Spark necesita que cada fila emitida por el RDD sea una tupla o estructura iterable para poder mapearla a columnas de un DataFrame.

Despues el .toDF(["target_id"]) convierte el RDD de tuplas (id_sample,) en un DataFrame de Spark con una sola columna llamada "target_id".

**El distinct() elimina IDs duplicados en caso de que un mismo id_sample haya sido identificado como el punto más cercano para dos centroides distintos. Esto evita multiplicar filas accidentalmente durante el join.**

In [43]:
target_ids_df = closest_per_cluster_rdd.map(lambda x: (x[1][0],)).toDF(["target_id"])

In [44]:
target_ids_df.count()

702

**Paso 2: Actualización de train_df mediante Left Join**

En primer lugar se realiza un Left Outer join manteniendo todas las filas de train_df:
- Si un id_sample de train_df coincide con target_ids_df, la columna auxiliar "target_id" tomará el valor del ID.
- Si un id_sample no coincide, la columna "target_id" valdrá null (None).

Despues .withColumn("state", sql_f.when(...).otherwise(...)) Sobreescribe (o crea) la columna "state" evaluando fila por fila:
- sql_f.when(sql_f.col("target_id").isNotNull(), sql_f.lit("L")): Si "target_id" no es nulo (significa que esa muestra era la más cercana a algún centroide), asigna el literal "L".
- .otherwise(sql_f.col("state")): Si "target_id" es nulo, conserva el valor original que tenía la columna "state"

Por último, se elimina: Elimina la columna temporal "target_id" utilizada para el cruce, dejando train_df exactamente con su esquema original pero con los estados actualizados.

In [45]:
# 2. Hacer Left Join y actualizar la columna 'state'
train_df = train_df.join(
    target_ids_df,
    train_df["id_sample"] == target_ids_df["target_id"],
    how="left"
).withColumn(
    "state",
    sql_f.when(sql_f.col("target_id").isNotNull(), sql_f.lit("L")).otherwise(sql_f.col("state"))
).drop("target_id")

In [46]:
# Se materializa la cache de train_df y se cuenta tanto labeled como unlabeled
stats = train_df.select(
    sql_f.count(sql_f.when(sql_f.col("state") == "L", True)).alias("labeled"),
    sql_f.count(sql_f.when(sql_f.col("state") == "U", True)).alias("unlabeled"),
).first()

# Extraer los valores
labeled_size = stats["labeled"]
unlabeled_size = stats["unlabeled"]
train_size = labeled_size + unlabeled_size
print(train_size)
print(labeled_size)
print(unlabeled_size)

70262
4192
66070


In [47]:
# Conjunto de datos etiquetado
labeled_df = train_df.filter(
    sql_f.col("state") == "L"
)

# Entrenar el modelo
lr_model = lr.fit(labeled_df)

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Realizar predicciones
test_predictions = lr_model.transform(test_df)

# Accuracy
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(test_predictions)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7909
